# VEP DNA Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True
    
import pandas as pd
import polars as pl
import seqpro as sp
import numpy as np
import pooch
from tqdm import tqdm
from pathlib import Path
from tempfile import TemporaryDirectory
import genvarloader as gvl

# Local code
import src.genvarloader as GVL
import src.vep_pipeline as vp
import src.utils as utils

# Set environment variable to suppress datetime warnings
os.environ['PYTHONWARNINGS'] = 'ignore::DeprecationWarning:jupyter_client.session'
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")

/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Create GVL database

## Download population data

In [2]:
# GRCh38 chromosome 22 sequence
reference = pooch.retrieve(
    url="https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz",
    known_hash="sha256:974f97ac8ef7ffae971b63b47608feda327403be40c27e391ee4a1a78b800df5",
    progressbar=True,
)
if not Path(f"{reference[:-3]}.bgz").exists():
    !gzip -dc {reference} | bgzip > {reference[:-3]}.bgz
reference = reference[:-3] + ".bgz"

# PLINK 2 files
variants = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pgen",
    known_hash="md5:31aba970e35f816701b2b99118dfc2aa",
    progressbar=True,
    fname="1kGP.chr22.pgen",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.psam",
    known_hash="md5:eefa7aad5acffe62bf41df0a4600129c",
    progressbar=True,
    fname="1kGP.chr22.psam",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pvar",
    known_hash="md5:5f922af91c1a2f6822e2f1bb4469d12b",
    progressbar=True,
    fname="1kGP.chr22.pvar",
)

# BED
bed_path = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/chr22_egenes.bed",
    known_hash="md5:ccb55548e4ddd416d50dbe6638459421",
    progressbar=True,
)


## Create BED file

This BED file is based on the coordinates of the UTR variants we're planning on injecting, surrounded by 500kb windows.

In [3]:
bed = gvl.read_bedlike("data/UTR/snps_500kb_windows.bed")[:20]
bed.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""22""",16835042,17335042,"""340557""",0.0,"""+"""
"""22""",16835050,17335050,"""340558""",0.0,"""+"""
"""22""",16835053,17335053,"""340559""",0.0,"""+"""
"""22""",16835059,17335059,"""894942""",0.0,"""+"""
"""22""",16835065,17335065,"""340560""",0.0,"""+"""


## Create GVL database

In [4]:
tmp_dir = TemporaryDirectory(suffix=".gvl")
ds_path = tmp_dir.name
gvl.write(
    path=ds_path,
    bed=gvl.with_length(bed, 2**17),  # change region length to 131,072 bp
    variants=variants,
    overwrite=True,
)

2025-05-17 13:20:25.435 | INFO     | genvarloader._dataset._write:write:76 - Writing dataset to /tmp/tmp6ugj8s6k.gvl
2025-05-17 13:20:25.435 | INFO     | genvarloader._dataset._write:write:83 - Found existing GVL store, overwriting.
2025-05-17 13:20:25.453 | INFO     | genoray._pgen:_read_index:1054 - Loading genoray index.
2025-05-17 13:20:26.129 | INFO     | genvarloader._dataset._write:write:151 - Using 451 samples.
2025-05-17 13:20:26.130 | INFO     | genvarloader._dataset._write:write:157 - Writing genotypes.
Processing genotypes for 20 regions on contig 22: 100%|██████████| 20/20 [00:01<00:00, 18.93 region/s]
2025-05-17 13:20:27.419 | INFO     | genvarloader._dataset._write:write:181 - Finished writing.


## Import GVL database

In [5]:
ds = (
    gvl.Dataset.open(ds_path, reference=reference)
    .with_seqs("haplotypes")
    .with_len(2**17)
)

2025-05-17 13:20:28.889 | INFO     | genvarloader._dataset._impl:open:215 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-17 13:20:28.942 | INFO     | genvarloader._dataset._reconstruct:from_path:298 - Loading variant data.
2025-05-17 13:20:28.979 | INFO     | genvarloader._dataset._impl:open:302 - Opened dataset:
GVL store at /tmp/tmp6ugj8s6k.gvl
Is subset: False
# of regions: 20
# of samples: 451
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: None
Tracks available: None



## Run VEP

Initialize xarray Dataset.

In [23]:
# Import sites_ds
sites = gvl.sites_vcf_to_table('./data/UTR/filtered_chr22_snps.vcf')
site_ds = gvl.DatasetWithSites(ds, sites) 
# Add the site_name column
GVL.add_site_name(site_ds)

# Create path for results file
results_dir = 'vep_results'
variant_set = "ClinVar_UTR"
zarr_path = os.path.join(results_dir, f"{variant_set}.zarr")
os.makedirs(results_dir, exist_ok=True)

# Initialize or load the dataset
models = ["flashzoi", "evo2-7b", "spliceai_mm"]
ds_results = vp.init_or_load_xarray_dataset(
    zarr_path=zarr_path,
    models=models, 
    site_ds=site_ds, 
    force=True
)
ds_results

[E::idx_find_and_load] Could not retrieve index file for './data/UTR/filtered_chr22_snps.vcf'
[E::idx_find_and_load] Could not retrieve index file for './data/UTR/filtered_chr22_snps.vcf'
[E::idx_find_and_load] Could not retrieve index file for 'data/UTR/filtered_chr22_snps.vcf'
Reading records: 0 record [00:00, ? record/s][W::vcf_parse] Contig '22' is not defined in the header. (Quick workaround: index the file with tabix.)
Reading records: 5529 record [00:00, 350068.79 record/s]


Initializing new dataset at vep_results/ClinVar_UTR.zarr


Creating xarray dataset with 3 model(s)


Saving xarray dataset to vep_results/ClinVar_UTR.zarr
xarray dataset saved to vep_results/ClinVar_UTR.zarr


<xarray.Dataset> Size: 222MB
Dimensions:      (site: 3419, sample: 451, ploid: 2, slot: 3)
Coordinates:
  * site         (site) <U27 369kB 'chr22:17019506-17150578_G_A' ... 'chr22:1...
  * sample       (sample) <U7 13kB 'HG00096' 'HG00097' ... 'NA20826' 'NA20828'
  * ploid        (ploid) int64 16B 0 1
  * slot         (slot) <U12 144B 'VEP' 'VEP_acceptor' 'VEP_donor'
Data variables:
    flashzoi     (site, sample, ploid, slot) object 74MB None nan ... nan nan
    evo2-7b      (site, sample, ploid, slot) object 74MB None nan ... nan nan
    spliceai_mm  (site, sample, ploid, slot) object 74MB nan None ... None None

Populate the xarray dataset

In [17]:
force = False
# Iterate over each site, which are centered on the SNPs
## Useing .rows only accesses the variants that overlap at least one region
models = ["spliceai_mm"]
all_samples = site_ds.dataset.samples
all_ploid = ds_results.coords["ploid"].values.tolist()
update_frequency = "ploid"
verbose = True

# Iterate over models
for model_name in models:
    
    # Load the model
    model_name = model_name.lower()
    model = vp.load_model(model_name)
    device  = utils.get_device()

    # Init the model
    if device.type=="GPU":
        model.to(device.type)
        model.eval()
    
    # Load the tokenizer
    tokenizer = vp.load_tokenizer(model_name)

    for site in tqdm(site_ds.rows.iter_rows(named=True),
                     total=site_ds.n_rows,
                     desc="Iterating over sites"):
        site_idx = site['site_idx']
        site_name = site["site_name"]
        
        # Skip the first row in sites (WT)
        if site_idx == 0:
            continue

        # Iterate over each sample
        for sample_idx, sample_name in tqdm(enumerate(all_samples),
                                            total=len(all_samples),
                                            desc="Iterating over samples",
                                            leave=False): 
            # Get region ID(s) that the site falls within
            # region_idx = site_ds.rows[sample_idx, "region_idx"]
            
            # Iterate over ploidy
            for ploid_idx in all_ploid:
                # Skip if the value is already set
                current_value = ds_results[model_name].sel(site=site_name,
                                                           sample=sample_name, 
                                                           ploid=ploid_idx, 
                                                        #    slot="VEP"
                                                           )
                if current_value.notnull().any() and not force:
                    continue  

                # Extract and convert sequences
                ## Get the wildtype (wt) sequence
                seq_wt = GVL.get_wt_haps(site_ds=site_ds, 
                                          sample_idx=sample_idx,
                                          ploid_idx=ploid_idx, 
                                          as_str=True)
                ## Get the mutated (mut) sequence
                seq_mut = GVL.get_mut_haps(site_ds=site_ds, 
                                           site_idx=site_idx,
                                           sample_idx=sample_idx,
                                           ploid_idx=ploid_idx, 
                                           as_str=True)           
                # Run the model
                if verbose:
                    print(f"Running VEP: {model_name}, {site_name}, {sample_name}, {ploid_idx}")
                vep = vp.run_vep(model_name=model_name,
                                 model=model,
                                 tokenizer=tokenizer,
                                 seq_wt=seq_wt, 
                                 seq_mut=seq_mut)
                
                # Store only the relevant VEP results
                for k,v in vep.items():
                    # Only assign valid slot types
                    if k in ds_results[model_name].coords['slot'].values:        
                        ds_results[model_name].loc[
                            dict(site=site_name,
                                sample=sample_name, 
                                ploid=ploid_idx, 
                                slot=k)
                                ] = v
                        
                # Save after each ploid is complete
                if update_frequency=="ploid":
                    vp.update_xarray_dataset(ds=ds_results, 
                                             zarr_path=zarr_path,
                                             verbose=verbose)
        
        # --- Limit the number of samples and sites to run for testing --- #
            if sample_idx>3:
                break
        if site_idx>2:
            break


Iterating over sites:   0%|          | 0/3420 [00:00<?, ?it/s]

Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00096, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00096, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00097, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00097, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00099, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00099, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00100, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_G_A, HG00100, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


...

Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00097, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00097, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00099, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00099, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00100, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00100, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00101, 0
Updating zarr ==> vep_results/ClinVar_UTR.zarr
Running VEP: spliceai_mm, chr22:17019506-17150578_C_G, HG00101, 1
Updating zarr ==> vep_results/ClinVar_UTR.zarr


Iterating over sites:   0%|          | 3/3420 [04:07<78:25:03, 82.62s/it]


In [22]:
#### Streamlined version of script above ###
# ds_results = vp.run_vep_pipeline(site_ds=site_ds, 
#                                  zarr_path=zarr_path,
#                                  models=["spliceai_mm"], 
#                                  sample_limit=3,
#                                  site_limit=2)

Get non-null subset of VEP results 

In [145]:
ds_vep =  ds_results.where(ds_results.notnull())
vep_df = ds_vep.to_dataframe().dropna(subset=["spliceai_mm"]).reset_index()
print(vep_df.shape)
vep_df.head()

(6984, 7)


,site,sample,ploid,slot,flashzoi,evo2-7b,spliceai_mm
0,chr22:17019506-17150578_G_A,HG00096,0,VEP_donor,NaN,NaN,0.000183
1,chr22:17019506-17150578_G_A,HG00096,0,VEP_acceptor,NaN,NaN,0.000113
2,chr22:17019506-17150578_G_A,HG00096,1,VEP_donor,NaN,NaN,0.000126
3,chr22:17019506-17150578_G_A,HG00096,1,VEP_acceptor,NaN,NaN,0.000009
4,chr22:17019506-17150578_G_A,HG00097,0,VEP_donor,NaN,NaN,0.000152


## Get the unique haplotypes

In [25]:
site_ds

In [ ]:
import awkward as ak
import numba as nb
import numpy as np
from attrs import define
from seqpro._ragged import Ragged
from genvarloader._dataset._reconstruct import Haps, V_IDX_TYPE
from genvarloader._dataset._impl import Dataset


@define
class UniqueInfo:
    """Class to store unique genotypes."""

    # sample, ploidy => ds[region_idx, sample_idx][ploid_idx] (ploidy length) => unique haplotype sequence
    first_idxs: Ragged[np.void]
    """First haplotype indices. Shape: (n_regions, ~n_unique)"""
    inverse_idxs: ak.Array  # total elements = regions * samples * ploidy
    """Inverse indices. Shape: (n_regions, ~n_unique, ~n_haps)"""
    counts: Ragged[np.uint32]
    """Counts. Shape: (n_regions, ~n_unique)"""

    @property
    def n_regions(self) -> int:
        """Number of regions."""
        return self.first_idxs.shape[0]

    @property
    def n_unique(self):
        """Number of unique genotypes."""
        return self.first_idxs.lengths


def unique(dataset: Dataset) -> UniqueInfo:
    if not isinstance(dataset._seqs, Haps):
        raise ValueError(
            "Dataset must have genotypes/haplotypes to compute unique haplotype information."
        )

    genos = dataset._seqs.genotypes
    regions = dataset._full_regions
    out_len = dataset.output_length
    first_idxs, inverse_idxs, counts = _unique_genotypes(
        genos.data, genos.offsets, regions, out_len
    )


@nb.njit(nogil=True, cache=True)
def _unique_genotypes(v_idxs, v_offsets, regions, out_len): 
    pass